In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"D:\Shikha\bias_project")
CENSUS   = BASE_DIR / "validation_data" / "census_acs" / "usa_00001.csv.gz"

# Columns we need (ignore the rest to save memory)
USECOLS = ["YEAR", "STATEFIP", "SEX", "AGE", "LABFORCE", "OCC1950", "PERWT"]

# Filters: working-age, in labor force, valid occupation
# LABFORCE: 2 = "in labor force"; OCC1950: 999 = N/A/missing, 970+ = non-occupation codes
def keep(df):
    return df[
        (df["AGE"].between(16, 64)) &
        (df["LABFORCE"] == 2) &
        (df["OCC1950"] < 970)          # drop N/A and special codes
    ]

# Stream in chunks, aggregate weighted person-counts by (year, state, sex, occ)
agg = {}
CHUNK = 500_000
rows_read = 0

print(f"Reading {CENSUS.name} in chunks of {CHUNK:,}...")
for chunk in pd.read_csv(CENSUS, usecols=USECOLS, chunksize=CHUNK,
                         compression="gzip"):
    rows_read += len(chunk)
    f = keep(chunk)
    # weighted counts: sum PERWT per (year, state, sex, occ)
    g = f.groupby(["YEAR", "STATEFIP", "SEX", "OCC1950"])["PERWT"].sum()
    for key, val in g.items():
        agg[key] = agg.get(key, 0) + val
    print(f"  read {rows_read:,} rows | kept-agg groups so far: {len(agg):,}", end="\r")

print(f"\n\nTotal rows read: {rows_read:,}")

# Convert aggregated dict to a tidy DataFrame
occ_counts = (pd.Series(agg).rename("weighted_count").reset_index())
occ_counts.columns = ["year", "statefip", "sex", "occ1950", "weighted_count"]
print(f"Aggregated table: {len(occ_counts):,} (year, state, sex, occ) cells")
print(f"Years present: {sorted(occ_counts['year'].unique())}")
print(occ_counts.head(10).to_string(index=False))

# Save the aggregated counts (small) so we never re-read the 388MB file
OUT = BASE_DIR / "validation_data" / "census_acs" / "occ_counts_by_state_year.csv"
occ_counts.to_csv(OUT, index=False)
print(f"\nSaved aggregated counts: {OUT}")

Reading usa_00001.csv.gz in chunks of 500,000...
  read 36,098,692 rows | kept-agg groups so far: 221,552

Total rows read: 36,098,692
Aggregated table: 221,552 (year, state, sex, occ) cells
Years present: [np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980), np.int64(1990), np.int64(2000), np.int64(2010), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
 year  statefip  sex  occ1950  weighted_count
 1920         1    1        0          705.95
 1920         1    1        1          205.14
 1920         1    1        3           99.63
 1920         1    1        7          199.26
 1920         1    1        9         2204.31
 1920         1    1       10           98.84
 1920         1    1       29           96.81
 1920         1    1       32          796.09
 1920         1    1       35          301.09
 1920         1    1       36           99.63

Saved aggregated counts: D:\Shikha\bias_projec

In [2]:
counts = pd.read_csv(BASE_DIR / "validation_data" / "census_acs" / "occ_counts_by_state_year.csv")

# Drop non-occupation codes (0 = N/A/none; keep real occupations only)
counts = counts[counts["occ1950"] > 0]

def duncan_index(group):
    # pivot: occupation x sex weighted counts
    piv = group.pivot_table(index="occ1950", columns="sex",
                            values="weighted_count", aggfunc="sum", fill_value=0)
    male   = piv.get(1, pd.Series(0, index=piv.index))
    female = piv.get(2, pd.Series(0, index=piv.index))
    if male.sum() == 0 or female.sum() == 0:
        return np.nan
    male_share   = male / male.sum()
    female_share = female / female.sum()
    return 0.5 * (male_share - female_share).abs().sum()

duncan = (counts.groupby(["year", "statefip"])
          .apply(duncan_index)
          .rename("duncan_index")
          .reset_index())

print(f"Duncan index computed for {len(duncan)} state-year cells")
print(f"\nMean Duncan by year (higher = more occupational segregation):")
print(duncan.groupby("year")["duncan_index"].agg(["mean", "min", "max", "count"]).round(3))

OUT = BASE_DIR / "outputs" / "validation" / "duncan_index_by_state_year.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
duncan.to_csv(OUT, index=False)
print(f"\nSaved: {OUT}")

Duncan index computed for 762 state-year cells

Mean Duncan by year (higher = more occupational segregation):
       mean    min    max  count
year                            
1920  0.703  0.470  0.805     51
1930  0.698  0.548  0.820     51
1940  0.673  0.553  0.783     49
1950  0.661  0.567  0.771     49
1960  0.664  0.567  0.780     51
1970  0.622  0.502  0.746     51
1980  0.561  0.403  0.702     52
1990  0.509  0.373  0.601     51
2000  0.473  0.312  0.544     51
2010  0.468  0.288  0.561     51
2020  0.439  0.280  0.573     51
2021  0.427  0.216  0.537     51
2022  0.423  0.252  0.507     51
2023  0.420  0.250  0.509     51
2024  0.413  0.221  0.502     51

Saved: D:\Shikha\bias_project\outputs\validation\duncan_index_by_state_year.csv


In [3]:
import pandas as pd
import numpy as np
from scipy import stats

EMB = BASE_DIR / "outputs" / "embeddings"

# Map corpus periods -> Duncan census years (the 3 overlap points)
PERIOD_TO_YEAR = {"hist_1920s": 1920, "modern_2020": 2020, "modern_2024": 2024}

# Per-state bias methods: (file, score column)
methods = {
    "w2v":     (EMB / "word2vec" / "weat6_scores_word2vec_v2.csv", "weat6_pairs_d"),
    "glove":   (EMB / "glove"    / "weat6_scores_glove_v2.csv",    "weat6_d"),
    "bert":    (EMB / "bert"     / "seat_scores_bert_v2.csv",      "seat_career_family"),
    "roberta": (EMB / "roberta"  / "seat_scores_roberta.csv",      "seat_career_family"),
}

# Duncan, keyed by state abbrev (it's STATEFIP -> need abbrev to match bias scores)
duncan = pd.read_csv(BASE_DIR / "outputs" / "validation" / "duncan_index_by_state_year.csv")

# STATEFIP -> 2-letter abbrev
FIPS_TO_ABBR = {
    1:'AL',2:'AK',4:'AZ',5:'AR',6:'CA',8:'CO',9:'CT',10:'DE',11:'DC',12:'FL',13:'GA',
    15:'HI',16:'ID',17:'IL',18:'IN',19:'IA',20:'KS',21:'KY',22:'LA',23:'ME',24:'MD',
    25:'MA',26:'MI',27:'MN',28:'MS',29:'MO',30:'MT',31:'NE',32:'NV',33:'NH',34:'NJ',
    35:'NM',36:'NY',37:'NC',38:'ND',39:'OH',40:'OK',41:'OR',42:'PA',44:'RI',45:'SC',
    46:'SD',47:'TN',48:'TX',49:'UT',50:'VT',51:'VA',53:'WA',54:'WV',55:'WI',56:'WY'
}
duncan["state"] = duncan["statefip"].map(FIPS_TO_ABBR)

print("=== Duncan Index vs Language Bias (3 overlap years, per state) ===")
print("Hypothesis: more occupational segregation (higher Duncan) <-> more language bias\n")
print(f"{'Method':<9} {'Period':<13} {'Year':>5} {'n':>4} {'Pearson r':>11} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 72)

rows = []
for name, (path, col) in methods.items():
    bias = pd.read_csv(path)
    for period, year in PERIOD_TO_YEAR.items():
        b = bias[bias["period"] == period][["state", col]].rename(columns={col: "bias"})
        d = duncan[duncan["year"] == year][["state", "duncan_index"]]
        m = b.merge(d, on="state").dropna()
        if len(m) < 4:
            print(f"{name:<9} {period:<13} {year:>5} {len(m):>4}  (too few states)")
            continue
        pr, pp = stats.pearsonr(m["bias"], m["duncan_index"])
        sr, sp = stats.spearmanr(m["bias"], m["duncan_index"])
        print(f"{name:<9} {period:<13} {year:>5} {len(m):>4} {pr:>11.3f} {pp:>8.3f} {sr:>10.3f} {sp:>8.3f}")
        rows.append({"method": name, "period": period, "year": year, "n": len(m),
                     "pearson_r": round(pr,4), "pearson_p": round(pp,4),
                     "spearman_r": round(sr,4), "spearman_p": round(sp,4)})

OUT = BASE_DIR / "outputs" / "validation" / "duncan_bias_correlation.csv"
pd.DataFrame(rows).to_csv(OUT, index=False)
print(f"\nSaved: {OUT}")
print("\nNote: demo-scale validation on 3 overlap years (1920, 2020, 2024). Duncan covers")
print("all 15 census years; only these 3 have a corpus partner until full-scale extraction.")

=== Duncan Index vs Language Bias (3 overlap years, per state) ===
Hypothesis: more occupational segregation (higher Duncan) <-> more language bias

Method    Period         Year    n   Pearson r        p   Spearman        p
------------------------------------------------------------------------
w2v       hist_1920s     1920    5      -0.681    0.205     -0.400    0.505
w2v       modern_2020    2020   30       0.030    0.875      0.029    0.878
w2v       modern_2024    2024    3  (too few states)
glove     hist_1920s     1920    5      -0.543    0.345     -0.300    0.624
glove     modern_2020    2020   32      -0.160    0.381     -0.223    0.220
glove     modern_2024    2024   44       0.110    0.476      0.097    0.532
bert      hist_1920s     1920    9      -0.091    0.815      0.133    0.732
bert      modern_2020    2020   50      -0.184    0.202     -0.187    0.193
bert      modern_2024    2024   49      -0.004    0.977      0.054    0.711
roberta   hist_1920s     1920    9      -